# Задание 4.2. Как провисает нить между опорами
**Зелёный и жёлтый уровни**

Однородная тяжёлая гибкая нерастяжимая нить удовлетворяет уравнению равновесия
$$y''=\frac{1}{a}\sqrt{1+(y')^2},\qquad a=\frac{H}{q}>0,$$
где $H$ — горизонтальная составляющая натяжения, $q$ — вес единицы длины нити. Вертикальная координата направлена вверх. Горизонтальное расстояние между опорами принято за единицу длины; $h$ — шаг по координате $x$.

- **Зелёный уровень, разделы 1–4:** при $a=1$ и опорах $(0,0)$, $(1,0)$ подбираем начальный наклон методом стрельбы. Сравниваем с точной цепной линией.
- **Жёлтый уровень, разделы 5–7:** при опорах $(0,0)$, $(1,0.2)$ и длине нити $1.2$ подбираем сразу наклон и параметр $a$ двумерным методом Ньютона. Проверяем параметры на более мелкой сетке и исследуем влияние длины на провисание и натяжение.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

a = 1.0
shoot_tol = 1e-14
counts = np.array([10, 20, 40, 80, 160])
colors = ["#b44e58", "#cf8c35", "#589465", "#397d9b", "#7458a0"]

## 1. Сведение к задаче Коши

Введём $p=y'$ и перепишем уравнение как систему:
$$
\begin{cases}
y'=p,\\
p'=\dfrac{\sqrt{1+p^2}}{a},
\end{cases}
\qquad
y(0)=0,\quad p(0)=s.
$$

При фиксированном начальном наклоне $s$ эту систему интегрируем от $x=0$ до $x=1$ классическим методом Рунге–Кутты четвёртого порядка (RK4). Для вектора $u=(y,p)$ и правой части $f(u)$ шаг имеет вид
$$
\begin{aligned}
k_1&=f(u_n),&
k_2&=f(u_n+hk_1/2),\\
k_3&=f(u_n+hk_2/2),&
k_4&=f(u_n+hk_3),\\
u_{n+1}&=u_n+\frac{h}{6}(k_1+2k_2+2k_3+k_4).
\end{aligned}
$$

Используем $N$ равных шагов: $h=1/N$. Сетка содержит $N+1$ узел, включая обе опоры.

В коде дополнительно считаем накопленную длину
$\ell'(x)=\sqrt{1+p(x)^2}$, $\ell(0)=0$. Она не влияет на уравнения для $y,p$ и понадобится в жёлтом уровне. Таким образом, состояние хранится как $(y,p,\ell)$.


In [ ]:
def rhs(state, a):
    p = state[1]
    arc_speed = np.sqrt(1.0 + p*p)
    return np.array([p, arc_speed/a, arc_speed])


def integrate(s, n, a=1.0):
    """Интегрирование при заданном наклоне s: возвращает x и тройки (y, p, ell)."""
    h = 1.0/n
    x = np.linspace(0.0, 1.0, n + 1)
    state = np.empty((n + 1, 3))
    state[0] = [0.0, s, 0.0]

    for i in range(n):
        u = state[i]
        k1 = rhs(u, a)
        k2 = rhs(u + h*k1/2, a)
        k3 = rhs(u + h*k2/2, a)
        k4 = rhs(u + h*k3, a)
        state[i + 1] = u + h*(k1 + 2*k2 + 2*k3 + k4)/6

    return x, state

## 2. Метод стрельбы

Левое граничное условие выполняется при любом $s$. Для выполнения правого условия нужно найти корень функции
$$F_h(s)=y_h(1;s).$$
Индекс $h$ подчёркивает, что значение на правой опоре вычисляется численно.

Начальный отрезок для поиска: $s\in[-1,0]$. При $s=-1$ конец полученного профиля расположен ниже правой опоры, а при $s=0$ — выше. Корень ищем **методом бисекции**: делим отрезок пополам и сохраняем ту половину, на концах которой невязка меняет знак.

Критерий остановки — $|F_h(s)|\le 10^{-14}$. Он выбран значительно меньше ожидаемых ошибок профиля. Максимальное число делений отрезка — 80. Точное решение при подборе наклона не используется.

In [ ]:
def shoot(n, a=1.0, tol=1e-14, max_iter=80):
    left, right = -1.0, 0.0
    f_left = integrate(left, n, a)[1][-1, 0]
    f_right = integrate(right, n, a)[1][-1, 0]
    if f_left*f_right >= 0:
        raise ValueError("На концах отрезка должны быть разные знаки невязки.")

    history = []
    for iteration in range(1, max_iter + 1):
        s = (left + right)/2
        x, state = integrate(s, n, a)
        residual = state[-1, 0]
        history.append((s, residual))

        if abs(residual) <= tol:
            return {
                "n": n, "h": 1.0/n, "s": s, "x": x, "state": state,
                "residual": residual, "iterations": iteration,
                "history": np.array(history),
            }

        if f_left*residual < 0:
            right = s
        else:
            left = s
            f_left = residual

    raise RuntimeError("Требуемая невязка не достигнута.")


runs = {int(n): shoot(int(n), a, shoot_tol) for n in counts}
main = runs[40]

for s in [-1.0, 0.0]:
    residual = integrate(s, main["n"], a)[1][-1, 0]
    print(f"Начальный наклон s = {s:4.1f}: F_h(s) = {residual:+.8f}")
print(f"\nДля N = {main['n']}, h = {main['h']:.3f}:")
print(f"Найденный наклон: s_h = {main['s']:.12f}")
print(f"Невязка на правой опоре: F_h(s_h) = {main['residual']:+.3e}")
print(f"Число итераций бисекции: {main['iterations']}")

In [ ]:
trial_slopes = [-1.0, -0.75, -0.5, -0.25, 0.0]
fig, ax = plt.subplots(figsize=(9, 5.4), constrained_layout=True)

for s, color in zip(trial_slopes, colors):
    x, state = integrate(s, main["n"], a)
    y = state[:, 0]
    ax.plot(x, y, color=color, lw=1.5, label=f"s = {s:g}")
    ax.plot(1.0, y[-1], "o", ms=5, color=color, mfc="white")

ax.plot(main["x"], main["state"][:, 0], color="#222222", lw=2.2,
        label=f"Найдено: s ≈ {main['s']:.6f}")
ax.scatter([0, 1], [0, 0], marker="s", s=55, color="black",
           label="Опоры", zorder=5)
ax.axhline(0, color="gray", lw=0.8, alpha=0.5)
ax.set(xlabel="x", ylabel="y", xlim=(-0.04, 1.04),
       title="Пристрелочные профили при h = 0.025")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
plt.show()

Пустые кружки показывают правые концы пробных профилей. Чёрный профиль получен после подбора наклона и проходит через обе опоры с заданной численной точностью.

## 3. Сравнение с точным решением

При $a=1$ точный профиль из условия:
$$
y_*(x)=\cosh\left(x-\frac12\right)-\cosh\left(\frac12\right).
$$
Действительно, $y_*'=\sinh(x-\frac12)$, а
$$
y_*''=\cosh\left(x-\frac12\right)
=\sqrt{1+\sinh^2\left(x-\frac12\right)}.
$$
Оба граничных условия также выполнены.

Из точного профиля получаем контрольные значения:
$$
s_*=-\sinh\left(\frac12\right),\qquad
x_{\min}=\frac12,\qquad
y_{\min}=1-\cosh\left(\frac12\right).
$$
Глубина провисания относительно линии опор равна $d=-y_{\min}>0$.

In [ ]:
def exact(x):
    return np.cosh(np.asarray(x) - 0.5) - np.cosh(0.5)


s_exact = -np.sinh(0.5)
sag_exact = np.cosh(0.5) - 1.0
best = runs[int(counts[-1])]
middle_index = best["n"]//2
sag_numeric = -best["state"][middle_index, 0]

print(f"Точный начальный наклон:     {s_exact:.12f}")
print(f"Наклон при N = {best['n']}:         {best['s']:.12f}")
print(f"Ошибка наклона:              {abs(best['s'] - s_exact):.3e}")
print(f"\nТочное провисание:           {sag_exact:.12f}")
print(f"Провисание при N = {best['n']}:     {sag_numeric:.12f}")

x_dense = np.linspace(0.0, 1.0, 1001)
fig, ax = plt.subplots(figsize=(10, 3.6), constrained_layout=True)
ax.plot(x_dense, exact(x_dense), color="#333333", lw=2,
        label="Точная цепная линия")
ax.plot(main["x"], main["state"][:, 0], "o", color=colors[3],
        ms=4, mfc="white", label=f"Стрельба + RK4, h = {main['h']:g}")
ax.scatter([0, 1], [0, 0], marker="s", s=55, color="black", zorder=5,
           label="Опоры")
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.annotate(f"d ≈ {sag_exact:.6f}",
            xy=(0.5, -sag_exact), xytext=(0.61, -0.075),
            arrowprops={"arrowstyle": "->", "color": "#555555"})
ax.set(xlabel="x", ylabel="y", xlim=(-0.04, 1.04), ylim=(-0.165, 0.055),
       title="Найденная форма нити и точное решение")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=3, fontsize=9)
plt.show()

На масштабе всей нити численные точки практически совпадают с точной кривой. Поэтому точность далее оценивается по разности решений.

## 4. Уменьшение шага и оценка точности

Для каждой сетки наклон подбирается **заново**, с одним и тем же допуском бисекции. Рассматриваются $N=10,\ 20,\ 40,\ 80,\ 160$, то есть $h=0.1,\ 0.05,\ 0.025,\ 0.0125,\ 0.00625$.

Отдельно вычисляются:

- невязка на правой опоре $R_h=|y_h(1)|$;
- ошибка профиля в каждом узле $e_h(x_i)=y_h(x_i)-y_*(x_i)$;
- максимальная узловая ошибка $E_h=\max_i|e_h(x_i)|$.

Максимум берётся по узлам расчётной сетки. Малое $R_h$ показывает точность решения уравнения стрельбы на выбранной сетке, но не заменяет оценку $E_h$.

Экспериментальный порядок при уменьшении шага вдвое:
$$
p_h=\log_2\frac{E_h}{E_{h/2}}.
$$
Если $E_h\approx C h^4$, то ошибка при делении шага пополам уменьшается приблизительно в 16 раз, а $p_h$ близок к 4.

In [ ]:
steps = 1.0/counts
max_errors = []
boundary_errors = []

for n in counts:
    run = runs[int(n)]
    run["error"] = run["state"][:, 0] - exact(run["x"])
    run["max_error"] = np.max(np.abs(run["error"]))
    max_errors.append(run["max_error"])
    boundary_errors.append(abs(run["residual"]))

max_errors = np.array(max_errors)
boundary_errors = np.array(boundary_errors)
orders = np.log2(max_errors[:-1]/max_errors[1:])

print(f"{'N':>5} {'h':>9} {'s_h':>17} {'R_h':>12} {'E_h':>12} {'Порядок':>9}")
for i, n in enumerate(counts):
    run = runs[int(n)]
    order_text = "—" if i == 0 else f"{orders[i-1]:.3f}"
    print(f"{int(n):5d} {run['h']:9.5f} {run['s']:17.12f} "
          f"{boundary_errors[i]:12.3e} {max_errors[i]:12.3e} {order_text:>9}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.5), constrained_layout=True)
for n, color in zip(counts, colors):
    run = runs[int(n)]
    error = np.abs(run["error"])
    # Нулевую ошибку нельзя отобразить на логарифмической оси.
    visible_error = np.where(error > 0, error, np.nan)
    ax.semilogy(run["x"], visible_error, color=color, lw=1.6,
                label=f"h = {run['h']:g}")

ax.set(xlabel="x", ylabel=r"$|y_h(x_i)-y_*(x_i)|$",
       title="Ошибка профиля на пяти сетках", xlim=(0, 1))
ax.grid(which="both", alpha=0.2)
ax.legend(ncol=2, fontsize=9)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.8), constrained_layout=True)
ax.loglog(steps, max_errors, "o-", color=colors[3], lw=1.8,
          label=r"Максимальная ошибка профиля $E_h$")
ax.loglog(steps, boundary_errors, "s-", color=colors[0], lw=1.2,
          label=r"Невязка на опоре $R_h$")
reference = max_errors[0]*(steps/steps[0])**4
ax.loglog(steps, reference, "--", color="#333333", alpha=0.75,
          label=r"Ориентир $C h^4$")
ax.axhline(shoot_tol, color="#999999", ls=":", label="Допуск бисекции")
ax.set(xlabel="Шаг h", ylabel="Абсолютная ошибка",
       title="Сходимость при уменьшении шага")
ax.grid(which="both", alpha=0.2)
ax.legend(fontsize=9)
plt.show()

Максимальная ошибка профиля уменьшается примерно в 16 раз при каждом делении шага пополам. Наклон зависимости $E_h$ от $h$ на графике с двумя логарифмическими осями близок к 4:
$$\log E_h\approx\log C+4\log h.$$
Это согласуется с четвёртым порядком метода RK4. Граничная невязка остаётся ниже допуска бисекции и не обязана монотонно уменьшаться вместе с шагом.





## Итог зелёного уровня

1. Методом стрельбы найден начальный наклон $s\approx-0.52109530549$. Полученный профиль симметричен относительно середины пролёта с точностью расчёта и совпадает с точной цепной линией. Минимальная высота равна примерно $-0.127625965206$, глубина провисания — $0.127625965206$ в единицах расстояния между опорами.
2. При уменьшении шага с $0.1$ до $0.00625$ максимальная узловая ошибка профиля уменьшается примерно с $1.97\cdot10^{-8}$ до $3.09\cdot10^{-13}$. Экспериментальные порядки близки к 4, что соответствует методу RK4.
3. На всех сетках невязка на правой опоре меньше $10^{-14}$. При этом на грубой сетке ошибка внутри пролёта существенно больше. Поэтому точность подбора начального наклона и точность интегрирования нужно контролировать раздельно.

## 5. Жёлтый уровень: две неизвестные при заданной длине

Теперь правая опора находится на высоте $b=0.2$, а длина нити равна $L=1.2$. Это больше расстояния между опорами $\sqrt{1+b^2}\approx1.0198$, поэтому нить может провисать.

Начальное состояние для стрельбы: $(y,p,\ell)(0)=(0,s,0)$. По заданным $s,a$ интегрируем систему из раздела 1 методом RK4. Подбор ведём по двум переменным $\boldsymbol z=(s,\alpha)$, где $\alpha=\ln a$. Тогда $a=e^\alpha>0$ во всех допустимых приближениях.

На правой границе сравниваем высоту и длину с заданными:
$$
F_h(\boldsymbol z)=
\begin{pmatrix}
R_y\\ R_\ell
\end{pmatrix}
=
\begin{pmatrix}
y_h(1;s,e^\alpha)-b\\
\ell_h(1;s,e^\alpha)-L
\end{pmatrix}.
$$
В методе Ньютона заменяем изменение невязки линейным приближением
$F_h(\boldsymbol z+\delta\boldsymbol z)\approx F_h(\boldsymbol z)+J_h\delta\boldsymbol z$.
Столбец $j$ матрицы $J_h$ вычисляем центральной разностью: меняем только переменную $z_j$ на $\pm d_j$, дважды интегрируем и делим разность невязок на $2d_j$. Берём $d_j=10^{-5}\max(1,|z_j|)$.

Затем решаем систему $J_h\delta\boldsymbol z=-F_h$. Пробуем полную поправку. Если максимум модулей двух невязок не уменьшается, делим поправку пополам до уменьшения. Так можно сделать первый шаг даже из приближения, для которого локальная линейная модель ещё неточна.

Начинаем с $s=-0.6$, $a=0.7$: нить выходит из левой опоры вниз, параметр $a$ имеет порядок длины пролёта. Точная форма в подборе не используется. Остановка: обе невязки по модулю не больше $10^{-13}$.

In [ ]:
height, length = 0.2, 1.2
initial_z = np.array([-0.6, np.log(0.7)])
newton_tol = 1e-13


def evaluate(z, n, b, L):
    slope, log_a = z
    tension_parameter = np.exp(log_a)
    x, states = integrate(slope, n, tension_parameter)
    residual = states[-1, [0, 2]] - np.array([b, L])
    return residual, x, states


def shooting_jacobian(z, n, b, L):
    columns = []
    for j in range(2):
        d = 1e-5*max(1.0, abs(z[j]))
        shift = np.zeros(2)
        shift[j] = d
        plus = evaluate(z+shift, n, b, L)[0]
        minus = evaluate(z-shift, n, b, L)[0]
        columns.append((plus-minus)/(2*d))
    return np.column_stack(columns)


def shoot_unequal(n, b, L):
    if L <= np.hypot(1.0, b):
        raise ValueError("Длина должна превышать расстояние между опорами.")
    z = initial_z.copy()
    residual, x, states = evaluate(z, n, b, L)
    history = []

    for k in range(25):
        norm = np.linalg.norm(residual, ord=np.inf)
        record = dict(k=k, z=z.copy(), residual=residual, x=x, states=states,
                      factor=None)
        history.append(record)
        if norm <= newton_tol:
            return history

        correction = np.linalg.solve(shooting_jacobian(z, n, b, L), -residual)
        for backtracks in range(13):
            factor = 2.0**(-backtracks)
            candidate = z + factor*correction
            with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                candidate_data = evaluate(candidate, n, b, L)
            candidate_residual = candidate_data[0]
            if (np.isfinite(candidate_residual).all()
                    and np.linalg.norm(candidate_residual, ord=np.inf) < norm):
                record["factor"] = factor
                z = candidate
                residual, x, states = candidate_data
                break
        else:
            raise RuntimeError("Не удалось уменьшить невязку.")
    raise RuntimeError("Ньютон не достиг заданного допуска.")


demo_n = 48
newton_history = shoot_unequal(demo_n, height, length)
print("γ — доля поправки, применённая при переходе к следующей строке.")
print(f"{'k':>3} {'s':>13} {'a':>13} {'R_y':>14} {'R_ℓ':>14} {'γ':>6}")
for row in newton_history:
    slope, log_a = row["z"]
    r_y, r_l = row["residual"]
    gamma_text = "—" if row["factor"] is None else f"{row['factor']:g}"
    print(f"{row['k']:3d} {slope:13.9f} {np.exp(log_a):13.9f} "
          f"{r_y:+14.5e} {r_l:+14.5e} {gamma_text:>6}")

In [ ]:
count = len(newton_history)
fig, axes = plt.subplots((count+2)//3, 3, figsize=(10.8, 5.8), squeeze=False,
                         sharex=True, sharey=True, constrained_layout=True)
y_lo = min(row["states"][:, 0].min() for row in newton_history)
y_hi = max(height, max(row["states"][:, 0].max() for row in newton_history))
for ax, row in zip(axes.flat, newton_history):
    profile = row["states"][:, 0]
    ax.plot([0, 1], [0, height], ":", color="0.65")
    ax.plot(row["x"], profile, lw=1.9, color=colors[3])
    ax.scatter([0, 1], [0, height], marker="s", color="#222222", s=27, zorder=4)
    ax.scatter(1, profile[-1], color=colors[0], s=22, zorder=5)
    ax.set(title=f"k = {row['k']}  ·  a = {np.exp(row['z'][1]):.4f}",
           xlabel="x", ylabel="y", xlim=(-0.03, 1.04), ylim=(y_lo-0.05, y_hi+0.04))
    ax.grid(alpha=0.18)
for ax in list(axes.flat)[count:]:
    ax.set_visible(False)
fig.suptitle(f"Профили по итерациям, N = {demo_n}: квадраты — опоры, круг — правый конец")
plt.show()

residual_history = np.abs(np.array([r["residual"] for r in newton_history]))
k = np.arange(count)
fig, ax = plt.subplots(figsize=(7.5, 4.1), constrained_layout=True)
for j, (color, name) in enumerate(zip([colors[3], colors[0]], [r"$|R_y|$", r"$|R_\ell|$"])):
    visible = np.where(residual_history[:, j] > 0, residual_history[:, j], np.nan)
    ax.semilogy(k, visible, ["o-", "s--"][j], color=color, lw=1.6, label=name)
ax.axhline(newton_tol, color="0.45", ls=":", label="Допуск")
ax.set(xlabel="Итерация k", ylabel="Модуль невязки", xticks=k,
       title="Ньютон: невязки правой высоты и длины")
ax.grid(which="both", alpha=0.2)
ax.legend()
plt.show()

Профили на рисунке показаны в одинаковом масштабе. Квадрат справа — заданная опора, цветной круг — конец текущего профиля. К последним итерациям формы визуально совпадают, поэтому дальнейшее улучшение заметнее на графике невязок. Знаки невязок сохранены в таблице: положительное значение означает слишком высокий конец или слишком длинную нить.

## 6. Проверка найденных параметров на мелкой сетке

Малая невязка Ньютона показывает, что условия хорошо выполнены **на сетке подбора**. Чтобы проверить параметры для исходного уравнения, повторим интегрирование с теми же $s,a$ на сетке $N_{\rm check}=3072$. Повторно подбирать их на проверочной сетке нельзя: тогда мы проверяли бы уже другой результат.

Для сравнения шагов выполняем подбор на сетках $N=24,48,96,192,384$. В таблице отдельно показываются невязка подбора и две ошибки проверочного расчёта. Уменьшение ошибок проверочного расчёта говорит об уточнении параметров при сгущении сетки подбора.

In [ ]:
fit_counts = [24, 48, 96, 192, 384]
check_n = 3072
fits, check_residuals = {}, {}
print(f"{'N подбора':>10} {'max|F_h|':>13} {'R_y, проверка':>16} {'R_ℓ, проверка':>16}")
for n in fit_counts:
    fit = newton_history[-1] if n == demo_n else shoot_unequal(n, height, length)[-1]
    fits[n] = fit
    check_residuals[n] = evaluate(fit["z"], check_n, height, length)[0]
    r_y, r_l = check_residuals[n]
    print(f"{n:10d} {np.max(np.abs(fit['residual'])):13.3e} "
          f"{r_y:+16.6e} {r_l:+16.6e}")

final_z = fits[fit_counts[-1]]["z"]
s_yellow, a_yellow = final_z[0], np.exp(final_z[1])
print(f"\nНайдено при N = {fit_counts[-1]}: s = {s_yellow:.11f}, a = {a_yellow:.11f}")

fig, ax = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
h_fit = 1.0/np.array(fit_counts)
check_errors = np.abs(np.stack(list(check_residuals.values())))
ax.loglog(h_fit, check_errors[:, 0], "o-", color=colors[3], label="Ошибка высоты")
ax.loglog(h_fit, check_errors[:, 1], "s--", color=colors[0], label="Ошибка длины")
ax.loglog(h_fit, check_errors[0, 0]*(h_fit/h_fit[0])**4, "k:", label=r"Ориентир $h^4$")
ax.set(xlabel="Шаг сетки подбора h", ylabel="Модуль проверочной невязки",
       title=f"Фиксированные s и a проверяются на N = {check_n}")
ax.legend()
ax.grid(which="both", alpha=0.2)
plt.show()

Дополнительно проверим, достаточно ли мелка сама проверочная сетка. При неизменных $s,a$ сравним интегрирование на $N=1536$ и $3072$ и результат по аналитической формуле для тех же параметров.

Из $p'/\sqrt{1+p^2}=1/a$ получаем $w(x)=\operatorname{arsinh}p(x)=\operatorname{arsinh}s+x/a$. Поэтому
$$
y(x)=a\,[\cosh w(x)-\cosh w(0)],\qquad
\ell(x)=a\,[\sinh w(x)-\sinh w(0)].
$$
Эта формула служит только проверкой: параметры по-прежнему найдены численным методом Ньютона.

In [ ]:
def exact_endpoint(s, a):
    w0 = np.arcsinh(s)
    w1 = w0 + 1/a
    return a*np.array([np.cosh(w1)-np.cosh(w0), np.sinh(w1)-np.sinh(w0)])


analytic_check = exact_endpoint(s_yellow, a_yellow)-np.array([height, length])
verification = [(f"RK4, N = {n}", evaluate(final_z, n, height, length)[0])
                for n in [1536, check_n]]
verification.append(("Аналитическая формула", analytic_check))
print(f"{'Проверка':>23} {'R_y':>16} {'R_ℓ':>16}")
for label, residual in verification:
    print(f"{label:>23} {residual[0]:+16.6e} {residual[1]:+16.6e}")

На сетке подбора $N=384$ параметры дают на более мелкой сетке ошибки порядка $10^{-12}$ по высоте и длине. Аналитическая проверка даёт те же значения с учётом округления. Значит, уменьшение шага только в проверочном интегрировании уже не устранит остаточную ошибку параметров: для этого нужно уточнять сам подбор.

## 7. Как длина влияет на провисание и натяжение

Оставим опоры на тех же местах и сравним $L=1.10,\ 1.20,\ 1.30$. Провисание измеряем относительно отрезка между опорами:
$$D=\max_{0\le x\le1}\bigl(bx-y(x)\bigr).$$
На сетке берём максимум по узлам. Это вертикальный зазор до наклонного отрезка; он отличается от глубины нижней точки относительно левой опоры.

При фиксированном $q$ горизонтальное натяжение задаётся $H=qa$. Чтобы найти модуль полного натяжения $\mathcal T(x)$, учитываем наклон: его вертикальная составляющая равна $Hp(x)$, поэтому
$$\frac{\mathcal T(x)}{q}=a\sqrt{1+p(x)^2}.$$
Числа приведены в виде $H/q$ и $\mathcal T/q$ в единицах длины пролёта: конкретное значение $q$ не задано.

In [ ]:
length_comparison = []
for L in [1.10, 1.20, 1.30]:
    z = final_z if np.isclose(L, length) else shoot_unequal(384, height, L)[-1]["z"]
    _, x, states = evaluate(z, check_n, height, L)
    a_L = np.exp(z[1])
    gap = height*x-states[:, 0]
    tension = a_L*np.sqrt(1+states[:, 1]**2)
    length_comparison.append(dict(L=L, a=a_L, gap=gap.max(), x=x,
                                  y=states[:, 0], tension=tension))

print(f"{'L':>7} {'H/q = a':>12} {'Провисание D':>15} "
      f"{'T(0)/q':>12} {'T(1)/q':>12}")
for r in length_comparison:
    print(f"{r['L']:7.2f} {r['a']:12.6f} {r['gap']:15.6f} "
          f"{r['tension'][0]:12.6f} {r['tension'][-1]:12.6f}")

fig, axes = plt.subplots(1, 2, figsize=(11.3, 4.4), constrained_layout=True)
axes[0].plot([0, 1], [0, height], ":", color="0.4", label="Отрезок между опорами")
for r, color in zip(length_comparison, [colors[0], colors[3], colors[4]]):
    label = f"L = {r['L']:.2f}"
    axes[0].plot(r["x"], r["y"], color=color, lw=1.8, label=label)
    axes[1].plot(r["x"], r["tension"], color=color, lw=1.8, label=label)
axes[0].scatter([0, 1], [0, height], marker="s", color="black", s=27, zorder=5)
axes[0].set(xlabel="x", ylabel="y", title="Форма при разных длинах")
axes[1].set(xlabel="x", ylabel=r"$\mathcal{T}(x)/q$", title="Модуль натяжения вдоль нити")
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)
plt.show()

**Почему закономерность сохраняется при изменении длины.** Из точных формул для концов следует
$$
L^2-b^2=\left[2a\sinh\!\left(\frac{1}{2a}\right)\right]^2.
$$
Это получается при вычитании квадратов разностей $\sinh$ и $\cosh$ из раздела 6. Выражение в квадратных скобках равно $\sinh v/v$ при $v=1/(2a)>0$. Оно растёт с $v$: $\sinh v/v=\int_0^1\cosh(vr)\,dr$. Значит, с ростом $L$ при неизменном $b$ параметр $a$ уменьшается, а вместе с ним уменьшается $H$ при фиксированном $q$.

При меньшем $a$ профиль лежит ниже. Действительно, если бы разность «профиль с меньшим $a$ минус профиль с большим $a$» была где-то положительна, она имела бы положительный максимум внутри пролёта. В этой точке наклоны равны, а уравнение нити даёт большую вторую производную для меньшего $a$. Тогда вторая производная разности положительна, что невозможно в максимуме. Поэтому при увеличении длины зазор относительно отрезка между опорами растёт. Модуль натяжения в отдельной точке зависит также от наклона, как видно из графика.

## Итог жёлтого уровня

- Для опор $(0,0)$, $(1,0.2)$ и длины $1.2$ двумерная стрельба методом Ньютона даёт $s\approx-0.96064928689$, $a\approx0.48947192539$.
- Проверка при неизменных параметрах на $N=3072$ даёт ошибки около $1.4\cdot10^{-12}$ по высоте и $8.7\cdot10^{-13}$ по длине. Эти значения нужно отличать от невязок на сетке подбора.
- При увеличении длины с $1.10$ до $1.30$ зазор до отрезка между опорами растёт примерно с $0.183$ до $0.362$, а $H/q$ уменьшается с $0.723$ до $0.398$. Для нити длиной $1.2$ максимальный зазор равен примерно $0.283$.